# MLOps: From Model Training to Production Deployment

**Companion notebook** to the MLOps tutorial. This notebook provides runnable Python implementations for:
- Model serialisation (pickle, joblib, ONNX, MLflow)
- FastAPI inference service (simulated)
- Docker integration patterns
- Kubernetes manifest generation
- MLflow tracking & registry
- Prometheus metric simulation
- Drift detection with Evidently & SHAP
- CI/CD pipeline simulation

---

## Setup

Install dependencies: `pip install -r requirements.txt`

In [ ]:
# Imports
import numpy as np
import pandas as pd
import pickle
import joblib
import tempfile
import os
import json
import time
from datetime import datetime
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split

In [ ]:
# Create a synthetic dataset matching the mlops-engineering project use case
np.random.seed(42)
n_samples = 1000

# WTI Oil Prices (feature)
oil_prices = np.random.normal(65, 15, n_samples)

# XOM Stock Price (target) — linear relationship with noise
xom_prices = 0.8 * oil_prices + 25 + np.random.normal(0, 3, n_samples)

X = oil_prices.reshape(-1, 1)
y = xom_prices

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
model = LinearRegression()
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
print(f"Model R²: {r2_score(y_test, y_pred):.4f}")
print(f"Model RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")
print(f"Coefficient: β₁ = {model.coef_[0]:.4f}")
print(f"Intercept: β₀ = {model.intercept_:.4f}")
print(f"Formula: XOM = {model.intercept_:.4f} + {model.coef_[0]:.4f} × WTI")

## 1. Model Serialisation Patterns

Models must be serialised after training and deserialised at inference time. We compare four approaches: pickle, joblib, ONNX, and MLflow.

In [ ]:
# 1.1 Pickle Serialisation (current project approach)
print("=== Pickle Serialisation ===")
with tempfile.NamedTemporaryFile(suffix=".pkl", delete=False) as f:
    pickle_path = f.name
    pickle.dump(model, f)
print(f"Model saved to: {pickle_path}")
print(f"File size: {os.path.getsize(pickle_path)} bytes")

# Load back
with open(pickle_path, "rb") as f:
    loaded_pickle = pickle.load(f)
pred = loaded_pickle.predict([[80.0]])
print(f"Pickle inference (oil=$80): XOM=${pred[0]:.2f}")

os.unlink(pickle_path)

In [ ]:
# 1.2 Joblib Serialisation (sklearn-preferred)
print("\n=== Joblib Serialisation ===")
with tempfile.NamedTemporaryFile(suffix=".joblib", delete=False) as f:
    joblib_path = f.name
    joblib.dump(model, joblib_path)
print(f"Model saved to: {joblib_path}")
print(f"File size: {os.path.getsize(joblib_path)} bytes")

loaded_joblib = joblib.load(joblib_path)
pred = loaded_joblib.predict([[80.0]])
print(f"Joblib inference (oil=$80): XOM=${pred[0]:.2f}")
os.unlink(joblib_path)

In [ ]:
# 1.3 ONNX Serialisation (cross-platform)
print("\n=== ONNX Serialisation ===")
try:
    from skl2onnx import convert_sklearn
    from skl2onnx.common.data_types import FloatTensorType
    
    initial_type = [("float_input", FloatTensorType([None, 1]))]
    onnx_model = convert_sklearn(model, initial_types=initial_type)
    
    with tempfile.NamedTemporaryFile(suffix=".onnx", delete=False) as f:
        onnx_path = f.name
        f.write(onnx_model.SerializeToString())
    print(f"Model saved to: {onnx_path}")
    print(f"File size: {os.path.getsize(onnx_path)} bytes")
    
    # Inference with ONNX Runtime
    import onnxruntime as ort
    session = ort.InferenceSession(onnx_path)
    input_name = session.get_inputs()[0].name
    onnx_pred = session.run(None, {input_name: np.array([[80.0]], dtype=np.float32)})[0]
    print(f"ONNX inference (oil=$80): XOM=${onnx_pred[0][0]:.2f}")
    os.unlink(onnx_path)
except ImportError:
    print("skl2onnx or onnxruntime not installed. Skipping ONNX example.")
    print("Install with: pip install skl2onnx onnxruntime")

In [ ]:
# 1.4 MLflow Serialisation (with registry metadata)
print("\n=== MLflow Serialisation ===")
try:
    import mlflow
    import mlflow.sklearn
    
    with tempfile.TemporaryDirectory() as tmpdir:
        mlflow.set_tracking_uri(f"file://{tmpdir}/mlruns")
        mlflow.set_experiment("serialisation-demo")
        
        with mlflow.start_run() as run:
            mlflow.log_param("model_type", "LinearRegression")
            mlflow.log_metric("r2", r2_score(y_test, y_pred))
            mlflow.log_metric("rmse", np.sqrt(mean_squared_error(y_test, y_pred)))
            mlflow.sklearn.log_model(model, "model")
            
            run_id = run.info.run_id
            print(f"MLflow Run ID: {run_id}")
            
            # Load from MLflow
            loaded_mlflow = mlflow.sklearn.load_model(f"runs:/{run_id}/model")
            mlflow_pred = loaded_mlflow.predict([[80.0]])
            print(f"MLflow inference (oil=$80): XOM=${mlflow_pred[0]:.2f}")
except ImportError:
    print("MLflow not installed. Skipping MLflow example.")
    print("Install with: pip install mlflow")

### Serialisation Comparison

| Method | File Size | Speed | Portability | Versioning |
|---|---|---|---|---|
| **pickle** | Small | Fast | Python-only | Manual |
| **joblib** | Small | Fast | Python-only | Manual |
| **ONNX** | Medium | Fastest | Any language | Manual |
| **MLflow** | Medium | Fast | Python + REST | Built-in registry |

## 2. FastAPI Inference Service (Simulated)

The mlops-engineering project uses FastAPI for model serving. Here we simulate the inference endpoint locally.

In [ ]:
# 2.1 Define the inference logic (same as app/src/main.py)
from pydantic import BaseModel

class PredictionInput(BaseModel):
    data: list

class PredictionOutput(BaseModel):
    prediction: float
    timestamp: str

# Simulate the predict function
def predict_oil_stock(oil_price):
    """Simulates the /predict endpoint logic."""
    start = time.time()
    
    # Model inference (same as loaded_model.predict([input_data.data])[0])
    pred = model.predict([[oil_price]])[0]
    
    elapsed = time.time() - start
    
    return {
        "prediction": round(pred, 2),
        "timestamp": datetime.utcnow().isoformat(),
        "latency_ms": round(elapsed * 1000, 2)
    }

# Test it
result = predict_oil_stock(85.0)
print("Inference result:")
print(json.dumps(result, indent=2))

In [ ]:
# 2.2 Simulate multiple requests with metrics
print("=== Simulating 10 Inference Requests ===\n")

latencies = []
predictions = []

for oil_price in np.random.uniform(40, 100, 10):
    result = predict_oil_stock(oil_price)
    latencies.append(result["latency_ms"])
    predictions.append(result["prediction"])
    print(f"  Oil=${oil_price:.1f} → XOM=${result['prediction']:.2f} (latency={result['latency_ms']:.2f}ms)")

print(f"\nSummary:")
print(f"  Mean latency: {np.mean(latencies):.2f}ms")
print(f"  P95 latency:  {np.percentile(latencies, 95):.2f}ms")
print(f"  Mean prediction: ${np.mean(predictions):.2f}")
print(f"  Prediction std:  ${np.std(predictions):.2f}")

## 3. Docker Integration Patterns

Docker containerises the model + service. Here we show how to build the Dockerfile programmatically (simulated).

In [ ]:
# 3.1 Generate Dockerfile content
dockerfile_content = '''FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY src/ ./src/
COPY data/regression_model.pkl ./data/

EXPOSE 5000

CMD ["uvicorn", "src.main:app", "--host", "0.0.0.0", "--port", "5000"]
'''

print("=== Dockerfile ===")
print(dockerfile_content)

# 3.2 Multi-stage build variant
multistage = '''# Stage 1: Build dependencies
FROM python:3.11-slim AS builder
WORKDIR /build
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt -t /deps

# Stage 2: Runtime
FROM python:3.11-slim
WORKDIR /app
COPY --from=builder /deps /usr/local/lib/python3.11/site-packages
COPY src/ ./src/
COPY data/regression_model.pkl ./data/
CMD ["uvicorn", "src.main:app", "--host", "0.0.0.0", "--port", "5000"]
'''

print("\n=== Multi-Stage Dockerfile ===")
print(multistage)

In [ ]:
# 3.3 Generate docker-compose.yaml programmatically
compose = {
    "version": "3.8",
    "services": {
        "ml_app": {
            "build": "./app",
            "ports": ["5000:5000"],
            "restart": "unless-stopped"
        },
        "prometheus": {
            "image": "prom/prometheus:latest",
            "ports": ["9090:9090"],
            "volumes": ["./prometheus/prometheus.yml:/etc/prometheus/prometheus.yml"]
        },
        "grafana": {
            "image": "grafana/grafana",
            "ports": ["3000:3000"],
            "depends_on": ["prometheus"]
        }
    }
}

print("=== docker-compose.yaml ===")
print(json.dumps(compose, indent=2))

## 4. Kubernetes for MLOps

Kubernetes manifests for deploying the inference service with auto-scaling and rolling updates.

In [ ]:
# 4.1 Generate Kubernetes Deployment manifest
deployment = {
    "apiVersion": "apps/v1",
    "kind": "Deployment",
    "metadata": {"name": "ml-inference", "labels": {"app": "ml-inference"}},
    "spec": {
        "replicas": 3,
        "strategy": {
            "type": "RollingUpdate",
            "rollingUpdate": {"maxUnavailable": 1, "maxSurge": 1}
        },
        "selector": {"matchLabels": {"app": "ml-inference"}},
        "template": {
            "metadata": {"labels": {"app": "ml-inference"}},
            "spec": {
                "containers": [{
                    "name": "inference",
                    "image": "ml-ops:latest",
                    "ports": [{"containerPort": 5000}],
                    "resources": {
                        "requests": {"cpu": "500m", "memory": "512Mi"},
                        "limits": {"cpu": "2", "memory": "2Gi"}
                    },
                    "livenessProbe": {
                        "httpGet": {"path": "/", "port": 5000},
                        "initialDelaySeconds": 30
                    },
                    "readinessProbe": {
                        "httpGet": {"path": "/docs", "port": 5000},
                        "initialDelaySeconds": 15
                    }
                }]
            }
        }
    }
}

print("=== Kubernetes Deployment ===")
print(json.dumps(deployment, indent=2))
print(f"\nReplicas: {deployment['spec']['replicas']}")
print(f"Strategy: {deployment['spec']['strategy']['type']}")

In [ ]:
# 4.2 Generate HorizontalPodAutoscaler
hpa = {
    "apiVersion": "autoscaling/v2",
    "kind": "HorizontalPodAutoscaler",
    "metadata": {"name": "ml-inference-hpa"},
    "spec": {
        "scaleTargetRef": {
            "apiVersion": "apps/v1",
            "kind": "Deployment",
            "name": "ml-inference"
        },
        "minReplicas": 2,
        "maxReplicas": 10,
        "metrics": [
            {"type": "Resource", "resource": {
                "name": "cpu",
                "target": {"type": "Utilization", "averageUtilization": 70}
            }},
            {"type": "Resource", "resource": {
                "name": "memory",
                "target": {"type": "Utilization", "averageUtilization": 80}
            }}
        ]
    }
}

print("=== HorizontalPodAutoscaler ===")
print(json.dumps(hpa, indent=2))
print(f"\nAuto-scaling: {hpa['spec']['minReplicas']} → {hpa['spec']['maxReplicas']} pods")
print("Trigger: CPU > 70% or Memory > 80%")

In [ ]:
# 4.3 Simulate canary deployment routing
print("=== Canary Deployment Simulation ===")
print()

# Simulating traffic split
total_requests = 1000
canary_weight = 0.10  # 10% to new model

to_canary = 0
to_stable = 0

for i in range(total_requests):
    if np.random.random() < canary_weight:
        to_canary += 1
    else:
        to_stable += 1

print(f"Total requests: {total_requests}")
print(f"Stable model (v1.0): {to_stable} ({to_stable/total_requests*100:.1f}%)")
print(f"Canary model (v1.1): {to_canary} ({to_canary/total_requests*100:.1f}%)")
print(f"\nCanary ingress annotation:")
print(f"  nginx.ingress.kubernetes.io/canary-weight: "{int(canary_weight*100)}"")

## 5. MLflow Experiment Tracking & Model Registry

MLflow tracks experiments, logs parameters/metrics/artifacts, and manages model versions through staging → production promotion.

In [ ]:
# 5.1 Simulate MLflow Tracking (without server)
print("=== MLflow Experiment Tracking (Simulated) ===\n")

# Create an in-memory experiment record
experiment = {
    "experiment_name": "stock-price-prediction",
    "run_id": "abc123def456",
    "parameters": {
        "model_type": "LinearRegression",
        "features": ["WTI_Oil"],
        "test_size": 0.2,
        "random_state": 42
    },
    "metrics": {
        "r2": round(r2_score(y_test, y_pred), 4),
        "rmse": round(np.sqrt(mean_squared_error(y_test, y_pred)), 4),
        "mae": round(np.mean(np.abs(y_test - y_pred)), 4)
    },
    "tags": {
        "version": "v1.0.0",
        "dataset": "synthetic-oil-stock-data",
        "author": "mlops-engineering"
    },
    "timestamp": datetime.utcnow().isoformat()
}

print(f"Experiment: {experiment['experiment_name']}")
print(f"Run ID:     {experiment['run_id']}")
print(f"\nParameters:")
for k, v in experiment['parameters'].items():
    print(f"  {k}: {v}")
print(f"\nMetrics:")
for k, v in experiment['metrics'].items():
    print(f"  {k}: {v}")
print(f"\nTags:")
for k, v in experiment['tags'].items():
    print(f"  {k}: {v}")

In [ ]:
# 5.2 MLflow Model Registry Promotion Flow
print("=== Model Registry Promotion ===\n")

stages = ["None", "Staging", "Production", "Archived"]
versions = [
    {"version": 1, "stage": "Production", "r2": 0.78, "date": "2025-01-15"},
    {"version": 2, "stage": "Staging",    "r2": 0.82, "date": "2025-03-01"},
    {"version": 3, "stage": "None",       "r2": 0.85, "date": "2025-04-10"},
]

print(f"{'Version':<10} {'Stage':<15} {'R²':<10} {'Date':<15} {'Action':<20}")
print("-" * 70)

for v in versions:
    if v["stage"] == "None":
        action = "Register → Staging?"
    elif v["stage"] == "Staging":
        action = "Validate → Production?"
    else:
        action = "Monitor drift"
    print(f"{v['version']:<10} {v['stage']:<15} {v['r2']:<10.4f} {v['date']:<15} {action:<20}")

print("\n--- Model Promotion Decision ---")
best_new = max(versions, key=lambda v: v["r2"] if v["stage"] != "Production" else 0)
current_prod = max(versions, key=lambda v: v["r2"] if v["stage"] == "Production" else 0)

print(f"Current Production: v{current_prod['version']} (R²={current_prod['r2']})")
print(f"Best candidate:     v{best_new['version']} (R²={best_new['r2']})")

if best_new["r2"] > current_prod["r2"]:
    print(f"\n✓ Promote v{best_new['version']} to Production (R² improvement: {best_new['r2'] - current_prod['r2']:+.4f})")
else:
    print(f"\n✗ No promotion needed (current production model is best)")

In [ ]:
# 5.3 MLflow Model Serving (simulated)
print("=== MLflow Model Serving ===\n")

# Simulate MLflow's built-in serving
def mlflow_model_server(model, port=5001):
    """Simulate mlflow models serve --port 5001"""
    print(f"MLflow Model Server starting on port {port}...")
    print(f"Endpoint: http://127.0.0.1:{port}/invocations")
    print(f"Model: LinearRegression (sklearn flavour)")
    print()
    
    test_inputs = [45.0, 65.0, 85.0, 105.0]
    print(f"{'Input (Oil $)':<15} {'Prediction (XOM $)':<20}")
    print("-" * 35)
    for oil in test_inputs:
        pred = model.predict([[oil]])[0]
        print(f"{oil:<15.1f} {pred:<20.2f}")

mlflow_model_server(model)

## 6. Prometheus Metrics Simulation

The project exposes Prometheus metrics from the FastAPI service. Here we simulate the same metrics programmatically.

In [ ]:
# 6.1 Simulate Prometheus metrics for the inference service
class SimulatedMetricsCollector:
    """Simulate the Prometheus metrics exposed by the FastAPI service."""
    
    def __init__(self):
        self.total_requests = 0
        self.successful_predictions = 0
        self.failed_predictions = 0
        self.latencies = []
        self.predictions = []
        self.cpu_usage = []
        self.memory_usage = []
    
    def record_prediction(self, latency_ms, prediction, success=True):
        self.total_requests += 1
        self.latencies.append(latency_ms)
        self.predictions.append(prediction)
        
        # Simulate CPU/memory (correlated with latency)
        self.cpu_usage.append(25 + np.random.uniform(-5, 15))
        self.memory_usage.append(512 + np.random.uniform(-50, 100))
        
        if success:
            self.successful_predictions += 1
        else:
            self.failed_predictions += 1
    
    def get_metrics(self):
        return {
            "total_requests": self.total_requests,
            "successful_predictions": self.successful_predictions,
            "failed_predictions": self.failed_predictions,
            "error_rate": self.failed_predictions / max(self.total_requests, 1),
            "latency_p50_ms": np.percentile(self.latencies, 50) if self.latencies else 0,
            "latency_p95_ms": np.percentile(self.latencies, 95) if self.latencies else 0,
            "latency_p99_ms": np.percentile(self.latencies, 99) if self.latencies else 0,
            "mean_prediction": np.mean(self.predictions) if self.predictions else 0,
            "cpu_usage_percent": np.mean(self.cpu_usage) if self.cpu_usage else 0,
            "memory_usage_mb": np.mean(self.memory_usage) if self.memory_usage else 0,
        }

# Simulate 1000 requests
metrics = SimulatedMetricsCollector()
for _ in range(1000):
    oil_price = np.random.uniform(40, 100)
    latency = 5 + np.random.exponential(10)  # ~15ms average
    is_success = np.random.random() > 0.02   # 98% success rate
    pred = model.predict([[oil_price]])[0]
    metrics.record_prediction(latency, pred, success=is_success)

print("=== Prometheus Metrics Snapshot ===")
for k, v in metrics.get_metrics().items():
    if "rate" in k or "p" in k[-2:]:
        print(f"  {k}: {v:.2%}" if "rate" in k else f"  {k}: {v:.2f}")
    else:
        print(f"  {k}: {v}")

In [ ]:
# 6.2 Visualise the metrics
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Latency distribution
axes[0, 0].hist(metrics.latencies, bins=50, edgecolor="black", alpha=0.7)
axes[0, 0].axvline(np.percentile(metrics.latencies, 50), color="green", linestyle="--", label=f"P50={np.percentile(metrics.latencies, 50):.1f}ms")
axes[0, 0].axvline(np.percentile(metrics.latencies, 95), color="orange", linestyle="--", label=f"P95={np.percentile(metrics.latencies, 95):.1f}ms")
axes[0, 0].axvline(np.percentile(metrics.latencies, 99), color="red", linestyle="--", label=f"P99={np.percentile(metrics.latencies, 99):.1f}ms")
axes[0, 0].set_title("Request Latency Distribution")
axes[0, 0].set_xlabel("Latency (ms)")
axes[0, 0].set_ylabel("Frequency")
axes[0, 0].legend()

# Prediction distribution
axes[0, 1].hist(metrics.predictions, bins=50, edgecolor="black", alpha=0.7, color="blue")
axes[0, 1].axvline(np.mean(metrics.predictions), color="red", linestyle="--", label=f"Mean=${np.mean(metrics.predictions):.1f}")
axes[0, 1].set_title("Prediction Value Distribution")
axes[0, 1].set_xlabel("Predicted XOM Price ($)")
axes[0, 1].set_ylabel("Frequency")
axes[0, 1].legend()

# CPU over time (sample)
axes[1, 0].plot(metrics.cpu_usage[:200], alpha=0.7)
axes[1, 0].axhline(np.mean(metrics.cpu_usage), color="red", linestyle="--", label=f"Mean: {np.mean(metrics.cpu_usage):.1f}%")
axes[1, 0].set_title("CPU Usage (last 200 requests)")
axes[1, 0].set_xlabel("Request #")
axes[1, 0].set_ylabel("CPU %")
axes[1, 0].legend()

# Error rate
error_rate = metrics.failed_predictions / metrics.total_requests
axes[1, 1].bar(["Success", "Failed"], [metrics.successful_predictions, metrics.failed_predictions], 
               color=["green", "red"], alpha=0.7)
axes[1, 1].set_title(f"Requests (Error Rate: {error_rate:.2%})")
axes[1, 1].set_ylabel("Count")
for i, v in enumerate([metrics.successful_predictions, metrics.failed_predictions]):
    axes[1, 1].text(i, v + 1, str(v), ha="center")

plt.tight_layout()
plt.show()

## 7. Drift Detection with Evidently & SHAP

Drift detection monitors whether the production data distribution differs from the training data distribution. This is critical for maintaining model quality.

In [ ]:
# 7.1 Statistical Drift Detection Functions

def calculate_psi(expected, actual, bins=10):
    """Population Stability Index."""
    breaks = np.percentile(expected, np.linspace(0, 100, bins + 1))
    breaks[0] = -np.inf
    breaks[-1] = np.inf
    expected_bins = np.clip(np.digitize(expected, breaks) - 1, 0, bins)
    actual_bins = np.clip(np.digitize(actual, breaks) - 1, 0, bins)
    psi = 0
    for i in range(bins):
        p_i = np.mean(expected_bins == i) + 1e-10
        q_i = np.mean(actual_bins == i) + 1e-10
        psi += (p_i - q_i) * np.log(p_i / q_i)
    return psi

# Test with simulated drift
train_oil = np.random.normal(60, 10, 1000)     # Training distribution
prod_oil_no_drift = np.random.normal(60, 10, 100)  # No drift
prod_oil_drift = np.random.normal(75, 15, 100)     # Drifted

print("=== Statistical Drift Detection ===")
print(f"PSI (no drift):   {calculate_psi(train_oil, prod_oil_no_drift):.4f} {'✓' if calculate_psi(train_oil, prod_oil_no_drift) < 0.1 else '✗'}")
print(f"PSI (with drift): {calculate_psi(train_oil, prod_oil_drift):.4f} {'✓' if calculate_psi(train_oil, prod_oil_drift) < 0.25 else '✗ ALERT'}")

In [ ]:
# 7.2 KS Test for Drift
from scipy.stats import ks_2samp

ks_stat_no, ks_p_no = ks_2samp(train_oil, prod_oil_no_drift)
ks_stat_drift, ks_p_drift = ks_2samp(train_oil, prod_oil_drift)

print("=== Kolmogorov-Smirnov Test ===")
print(f"KS (no drift):   stat={ks_stat_no:.4f}, p={ks_p_no:.4f} {'✓' if ks_p_no > 0.05 else '✗'}")
print(f"KS (with drift): stat={ks_stat_drift:.4f}, p={ks_p_drift:.4f} {'✓' if ks_p_drift > 0.05 else '✗ ALERT'}")

# Visualise drift
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(train_oil, bins=30, alpha=0.6, label="Training", density=True)
axes[0].hist(prod_oil_no_drift, bins=30, alpha=0.6, label="Production (no drift)", density=True)
axes[0].set_title("No Drift: Distributions Match")
axes[0].legend()

axes[1].hist(train_oil, bins=30, alpha=0.6, label="Training", density=True)
axes[1].hist(prod_oil_drift, bins=30, alpha=0.6, label="Production (drifted)", density=True)
axes[1].set_title("Drift Detected: Distributions Diverge")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# 7.3 Production Drift Monitor Class
class ProductionDriftMonitor:
    """End-to-end drift monitoring for production ML."""
    
    def __init__(self, model, reference_data, feature_names, batch_size=100):
        self.model = model
        self.reference_data = reference_data
        self.feature_names = feature_names
        self.buffer = []
        self.batch_size = batch_size
        self.drift_history = []
        self.prediction_history = []
    
    def predict_and_monitor(self, features):
        """Make prediction and accumulate drift data."""
        # Run inference
        prediction = self.model.predict([features])[0]
        self.prediction_history.append(prediction)
        
        # Buffer features for batch drift check
        self.buffer.append(features)
        
        # Periodic drift check
        alert = False
        if len(self.buffer) >= self.batch_size:
            alert = self._run_drift_check()
        
        return {"prediction": prediction, "drift_alert": alert}
    
    def _run_drift_check(self):
        """Run drift checks on buffered production data."""
        current = np.array(self.buffer)
        
        # PSI for each feature
        drifts = []
        for i, name in enumerate(self.feature_names):
            psi = calculate_psi(self.reference_data[:, i], current[:, i])
            drifts.append({"feature": name, "PSI": psi, "alert": psi > 0.25})
        
        # Prediction drift
        ref_preds = self.model.predict(self.reference_data)
        curr_preds = self.model.predict(current)
        pred_psi = calculate_psi(ref_preds, curr_preds)
        
        result = {
            "timestamp": datetime.utcnow().isoformat(),
            "feature_drifts": drifts,
            "prediction_drift_psi": pred_psi,
            "prediction_drift_alert": pred_psi > 0.25,
            "overall_alert": any(d["alert"] for d in drifts) or pred_psi > 0.25
        }
        
        self.drift_history.append(result)
        self.buffer = []
        
        # Print alert
        if result["overall_alert"]:
            print(f"⚠ DRIFT ALERT at {result['timestamp']}")
            for d in drifts:
                if d['alert']:
                    print(f"  Feature '{d['feature']}': PSI={d['PSI']:.4f}")
            if result['prediction_drift_alert']:
                print(f"  Prediction distribution: PSI={pred_psi:.4f}")
        
        return result["overall_alert"]
    
    def get_drift_report(self):
        """Return a summary of all drift checks."""
        if not self.drift_history:
            return {"message": "No drift checks performed yet"}
        
        alerts = sum(1 for h in self.drift_history if h["overall_alert"])
        return {
            "total_checks": len(self.drift_history),
            "total_alerts": alerts,
            "alert_rate": alerts / len(self.drift_history),
            "predictions_monitored": len(self.prediction_history),
            "last_check": self.drift_history[-1]["timestamp"] if self.drift_history else None
        }

# 7.4 Test the monitor with simulated production data
print("=== Production Drift Monitor Test ===\n")

# Create reference data (training distribution)
ref_data = np.random.normal(60, 10, (1000, 1))

# Initialise monitor
monitor = ProductionDriftMonitor(
    model=model,
    reference_data=ref_data,
    feature_names=["WTI_Oil"],
    batch_size=50
)

# Simulate 200 production requests (first 100 normal, last 100 drifted)
print("Phase 1: Normal production data (no drift expected)")
for i in range(100):
    features = [np.random.normal(60, 10)]  # same as training
    result = monitor.predict_and_monitor(features)

print(f"\nPhase 2: Drifted production data (alerts expected)")
for i in range(100):
    features = [np.random.normal(80, 18)]  # drifted distribution
    result = monitor.predict_and_monitor(features)

print(f"\n=== Drift Monitor Summary ===")
report = monitor.get_drift_report()
for k, v in report.items():
    if "rate" in k:
        print(f"  {k}: {v:.2%}")
    else:
        print(f"  {k}: {v}")

In [ ]:
# 7.5 SHAP for Model Interpretability
print("=== SHAP Model Explanation ===\n")

try:
    import shap
    
    # Create SHAP explainer
    X_train_df = pd.DataFrame(X_train, columns=["WTI_Oil"])
    explainer = shap.Explainer(model, X_train_df)
    shap_values = explainer(X_train_df[:100])
    
    # Summary plot
    shap.summary_plot(shap_values, X_train_df[:100], show=False)
    plt.title("SHAP Feature Importance — Production Monitoring")
    plt.tight_layout()
    plt.show()
    
    # Waterfall for a single prediction
    single = X_train_df.iloc[[0]]
    sv_single = explainer(single)
    shap.waterfall_plot(sv_single[0], max_display=10, show=False)
    plt.title(f"SHAP Explanation: Oil=${single['WTI_Oil'].values[0]:.1f}")
    plt.tight_layout()
    plt.show()
    
    print("SHAP explanations generated successfully.")
    print("Expected value (baseline): ${:.2f}".format(explainer.expected_value))
    print("Actual prediction: ${:.2f}".format(model.predict(single)[0]))
    
except ImportError:
    print("SHAP not installed. Install with: pip install shap")

## 8. CI/CD Pipeline Simulation

Continuous integration validates model quality before deployment. Continuous deployment automates the rollout.

In [ ]:
# 8.1 Model Validation Gates (CI checks)
print("=== CI Validation Gates ===\n")

gates = {
    "R² ≥ 0.70": {"value": r2_score(y_test, y_pred), "pass": False, "threshold": 0.70},
    "RMSE ≤ 15.0": {"value": np.sqrt(mean_squared_error(y_test, y_pred)), "pass": False, "threshold": 15.0},
    "MAE ≤ 10.0": {"value": np.mean(np.abs(y_test - y_pred)), "pass": False, "threshold": 10.0},
    "No data drift (PSI < 0.25)": {"value": calculate_psi(X_train.flatten(), X_test.flatten()), "pass": False, "threshold": 0.25},
}

all_pass = True
for gate_name, gate in gates.items():
    if gate_name.startswith("R²"):
        gate["pass"] = gate["value"] >= gate["threshold"]
    else:
        gate["pass"] = gate["value"] <= gate["threshold"]
    
    status = "✓ PASS" if gate["pass"] else "✗ FAIL"
    print(f"  {status}: {gate_name}")
    print(f"         Value: {gate['value']:.4f}, Threshold: {gate['threshold']}")
    
    if not gate["pass"]:
        all_pass = False

print(f"\nOverall: {'✓ ALL GATES PASSED — Ready for deployment' if all_pass else '✗ GATES FAILED — Blocking deployment'}")

In [ ]:
# 8.2 Simulate CI/CD Pipeline
print("=== CI/CD Pipeline Execution ===\n")

pipeline_steps = [
    ("1. Code Push", "github.com/org/mlops-engineering", "✅ Done"),
    ("2. Lint & Format Check", "flake8, black", "✅ Done"),
    ("3. Unit Tests", "pytest app/tests/", "✅ Done"),
    ("4. Model Validation Gates", "R², RMSE, Drift checks", "✅ Done"),
    ("5. Build Docker Image", "docker build -t ml-ops:sha-abc123", "✅ Done"),
    ("6. Push to Registry", "docker push registry.example.com/ml-ops", "✅ Done"),
    ("7. Deploy to Staging", "kubectl apply -f k8s/staging/", "✅ Done"),
    ("8. Smoke Tests", "curl /predict with test data", "✅ Done"),
    ("9. Promote to Production", "kubectl apply -f k8s/production/", "⏳ In Progress"),
    ("10. Rollout Verification", "Monitor error rate & latency", "⏳ Pending"),
]

print(f"{'Step':<35} {'Detail':<40} {'Status':<15}")
print("-" * 90)
for step, detail, status in pipeline_steps:
    print(f"{step:<35} {detail:<40} {status:<15}")

print(f"\nAutomated via GitHub Actions (.github/workflows/deploy.yml)")
print("Deployment strategy: Rolling update (max unavailable: 1, max surge: 1)")

## 9. Full Architecture Summary

The complete MLOps stack demonstrated in this notebook:

In [ ]:
# 9.1 Architecture component inventory
architecture = {
    "Model Training": {
        "tool": "scikit-learn",
        "algorithm": "LinearRegression",
        "use_case": "XOM stock price from WTI oil price"
    },
    "Model Serialisation": {
        "options": ["pickle", "joblib", "ONNX", "MLflow"],
        "current": "pickle (regression_model.pkl)"
    },
    "Inference API": {
        "framework": "FastAPI",
        "endpoint": "POST /predict",
        "port": 5000,
        "docs": "/redoc"
    },
    "Containerisation": {
        "tool": "Docker",
        "base_image": "python:3.11-slim",
        "orchestration": "docker-compose → Kubernetes"
    },
    "Monitoring": {
        "metrics": ["total_requests", "latency_histogram", "prediction_gauge", "cpu/memory"],
        "storage": "Prometheus (port 9090)",
        "dashboard": "Grafana (port 3000)"
    },
    "Drift Detection": {
        "methods": ["PSI", "KS-test", "Evidently AI", "SHAP"],
        "frequency": "batch (every 100 requests)",
        "alerting": "Prometheus alertmanager"
    },
    "ML Lifecycle": {
        "tracking": "MLflow experiments",
        "registry": "MLflow Model Registry",
        "promotion": "None → Staging → Production"
    },
    "CI/CD": {
        "platform": "GitHub Actions",
        "gates": ["tests", "model validation", "drift check"],
        "deployment": "Rolling update on Kubernetes"
    }
}

print("=== MLOps Architecture Stack ===\n")
for component, details in architecture.items():
    print(f"{component}:")
    if isinstance(details, dict):
        for k, v in details.items():
            if isinstance(v, list):
                print(f"  {k}: {', '.join(str(x) for x in v)}")
            else:
                print(f"  {k}: {v}")
    print()

In [ ]:
# 9.2 End-to-End Inference Test
print("=== End-to-End Inference Test ===\n")

test_cases = [
    {"oil": 45.0, "expected_range": (50, 70)},
    {"oil": 65.0, "expected_range": (65, 85)},
    {"oil": 85.0, "expected_range": (80, 100)},
    {"oil": 105.0, "expected_range": (95, 120)},
]

all_pass = True
for tc in test_cases:
    oil = tc["oil"]
    pred = model.predict([[oil]])[0]
    in_range = tc["expected_range"][0] <= pred <= tc["expected_range"][1]
    status = "✓" if in_range else "✗"
    print(f"{status} Oil=${oil:.1f} → XOM=${pred:.2f} (expected: ${tc['expected_range'][0]}-${tc['expected_range'][1]})")
    if not in_range:
        all_pass = False

print(f"\nOverall: {'✓ ALL TESTS PASSED' if all_pass else '✗ SOME TESTS FAILED'}")
print(f"Model formula: XOM = {model.intercept_:.4f} + {model.coef_[0]:.4f} × WTI")